#### Assignment 1 (NLU 22)
##### Sentiment analysis using logistic regression

**Marks = 100**

In this assignment you train a binary classifier for classifying movie reviews from IMDB database. The task is to classify them as positive or negative. The task has been divided into several subtasks.

1. You will need the following libraries. Do not import everything.  
    * re
    * pandas
    * numpy
    * scipy
    * nltk
    * scikit-learn
2. The data consists of two directories--positive and negative reviews.   
3. Each review is a text document.
4. Stopwords need not be removed. In fact, for some "negative" stopwords you are required to modify the tokens.
5. You will have to map each document (email) to a vector (**tf/idf**) .
6. You should create the tf-idf vectors from scratch. <span style="color:red">Do **not** use library functions</span>.
7. Once you have the vectors for each document apply logistic regression to the training set to fix the weights. You *may* use **sklearn** logistic regression function from linear models.  
8. Test your model with the test set and report accuracy, recall and precision.
9. Write a short report (about 250 words) on the model and specifically about any issues with the data.
10. You will be provided with a set of functions. Your task is to complete them.
11. Do not change anything in the structure of these functions. If you have print functions for testing comment them out. You may write your "helper" functions, but must clearly and concisely state their purpose.
12. Commenting your code is important. But not too much commenting. **Ten percent** of marks is reserved for code efficiency, style and planning.
13. The following are the basic steps:
    1. *process the dataset*
    2. *build a vocabulary of the training set*
    3. *convert documents to vectors by* **tf/idf** *weighting*
    4. *each document will be represented by a vector of dimension equal to the size of the vocabulary* (lots of 0's!)
    5. train the model (use logistic regression from sk-learn linear models)
    6. test the model and compute performance measures
    7. write a short report at the end of the notebook
14. The dataset provided is relatively large (over 25MB). If you are using a slower computer test your code with slice of the dataset.

Read relevant parts in Jurafsky-Martin book (p113, 2025 edition) and lecture notes.
**Please follow the instructions**. Otheriwse, your code may not run properly and you may lose marks.

In [1]:
!pip install tqdm

In [2]:
import os, re
import numpy as np
import pandas as pd
import sys, shutil
import random
from scipy.stats import bernoulli
import time
import zipfile
from tqdm.auto import tqdm
from pathlib import Path
from collections import defaultdict, Counter
import multiprocessing as mp
import scipy.sparse as sp
from sklearn.preprocessing import MaxAbsScaler

In [3]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

In [4]:
def extract_imdb(zip_path, dest):
  """Extracts the IMDB dataset zip file to destination folder
  """
  zip_path = Path(zip_path)
  dest = Path(dest).resolve()

  if not zip_path.exists():
    raise FileNotFoundError(f"Zip not found: {zip_path}")

  if not zipfile.is_zipfile(zip_path):
    raise ValueError(f"File is not a valid zip archive: {zip_path}")

  out_dir = dest / zip_path.stem

  if out_dir.exists():
    n = sum(1 for _ in out_dir.rglob("*") if _.is_file())
    if n > 0:
      print(f"Already extracted ({n:,} files). Skipping. -> {out_dir}")
      return out_dir

  out_dir.mkdir(parents=True, exist_ok=True)

  with zipfile.ZipFile(zip_path, "r") as zf:
    members = zf.infolist()

    # Implement robust zip-slip guard
    for member in members:
      target = (dest / member.filename).resolve()
      if not str(target).startswith(str(dest)):
        raise RuntimeError(f"Zip-slip detected, aborting: {member.filename}")

    for member in tqdm(members, desc="Extracting", unit="file"):
      zf.extract(member, dest)

  file_count = sum(1 for _ in out_dir.rglob("*") if _.is_file())
  print(f"Done. Extracted {file_count:,} files -> {out_dir}")
  return out_dir

imdb_dataset = extract_imdb("imdb_dataset.zip", ".")

Already extracted (42,927 files). Skipping. -> /content/imdb_dataset


In [5]:
fl_tree = list(os.walk("imdb_dataset"))
data_dirs = [fl_tree[1][0], fl_tree[2][0]]
data_dirs

['imdb_dataset/pos', 'imdb_dataset/neg']

In [6]:
files_neg = fl_tree[1][2] #files of negative reviews
files_pos = fl_tree[2][2] #files of positive reviews

##### Task 1: create train-test split   **[10 marks]**  
1. This is a large dataset. So it is better to avoid loading the whole dataset into memory.
2. We only keep track of the file lists.
3. Note that there are two file lists, one for positive sentiment and the other for negative.
4. Combine them, shuffle the resulting single list, *after* storing the information about the sentiment. You could use a tuple.
5. The function below, that you complete, has a single input and two outputs. The input, *r*, represents the (approximate) fraction of files for the training. Rest are for testing. For example, if $r=0.8$, about 80% of the dataset is used for training. The output, *train_list* and *test_list*, are two lists of file names.
6. You will use these lists later to read the files from disk. Note that, you have to provide the full path.
7. Keep the directory "**imdb_dataset**" in the same directory as the notebook.
8. Do **not** change the directory structure.


In [7]:
def train_test_split(r):
    """Create a stratified random train/test split from the IMDB file lists.
    """
    import random

    # Pair filename with its sentiment label
    neg_reviews = [(fname, 0) for fname in files_neg] # 0 = negative
    pos_reviews = [(fname, 1) for fname in files_pos] # 1 = positive

    # Combine and shuffle in place adding seed for reproducability
    combined_reviews = neg_reviews + pos_reviews
    random.seed(42)
    random.shuffle(combined_reviews)

    # Slice based on fraction r
    split_idx = int(len(combined_reviews) * r)
    train_list = combined_reviews[:split_idx]
    test_list = combined_reviews[split_idx:]

    return train_list, test_list

In [8]:
tr3, ts3 = train_test_split(.8)
# print(f"Train: {len(tr3)} files, Test: {len(ts3)} files")

In [9]:
def get_path(fname, label):
  """Resolve the full path of a file from a (filename, label) tuple.
  """
  return os.path.join(data_dirs[label], fname)

In [10]:
#Identify the negative tokens. You will complete the function "neg_modifier" using these.
#You may add more but justify it if you do so.
import nltk
nltk.download('stopwords')

def neg_tokens():
    from nltk.corpus import stopwords
    stopWords = sorted(list(stopwords.words('english')))
    neg_pat = re.compile(r"\w+(n\'t)|no|not|never")
    neg_tok = defaultdict(int)
    for w in stopWords:
        if neg_pat.fullmatch(w):
            neg_tok[w] = 1
    return neg_tok

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [11]:
neg_w = neg_tokens()

##### Task 2: modify the documents containing negative tokens   **[15 marks]**  
1. Negative words are often used to express the opposite sentiment. Consider the following.
    1. *The movie wasn't bad*.
    2. *This book is no good*.
2. So we use a simple trick to create new tokens in cases like these.
3. Use the following construction as suggested in Jurafsky-Martin book (2025 edition).    
"*A very simple baseline that is commonly used in sentiment analysis to deal with
negation is the following: during text normalization, prepend the prefix NOT to
every word after a token of logical negation (n’t, not, no, never) until the next punc-
tuation mark. Thus the phrase     
'didn’t like this movie , but I'   
becomes    
'didn’t NOT_like NOT_this NOT_movie , but I'*"
4. Use **re** to to look for appropriate characters and use *re.split*.
5. The output are the tokens in the document, including the (*NOT*) modified tokens.
6. This will beused in subsequent functions.


In [12]:
def neg_modifier(txt):
    """Tokenise text and prepend NOT_ to every word following a negation token,
       up to the next punctuation mark (resets negation scope).
    """
    txt = txt.lower()
    punct = re.compile(r"[\.,\?;:!]")
    sp_ls = punct.split(txt)
    tok_ls = []

    for clause in sp_ls:
        tokens = clause.split()
        negating = False

        for tok in tokens:
            if tok in neg_w:
                tok_ls.append(tok)
                negating = True
            elif negating:
                tok_ls.append("NOT_" + tok)
            else:
                tok_ls.append(tok)

    return tok_ls

In [13]:
st = """Meanwhile, Tehran says it has no plans for the next round of peace talks in Islamabad.
What is said publicly doesn’t always reflect how everything is playing out behind the scenes.
Iran and the US both have their domestic audiences in mind, and will also want to apply public pressure on each other."""
cn = Counter(neg_modifier(st))
cn.items()

dict_items([('meanwhile', 1), ('tehran', 1), ('says', 1), ('it', 1), ('has', 1), ('no', 1), ('NOT_plans', 1), ('NOT_for', 1), ('NOT_the', 1), ('NOT_next', 1), ('NOT_round', 1), ('NOT_of', 1), ('NOT_peace', 1), ('NOT_talks', 1), ('NOT_in', 1), ('NOT_islamabad', 1), ('what', 1), ('is', 2), ('said', 1), ('publicly', 1), ('doesn’t', 1), ('always', 1), ('reflect', 1), ('how', 1), ('everything', 1), ('playing', 1), ('out', 1), ('behind', 1), ('the', 2), ('scenes', 1), ('iran', 1), ('and', 2), ('us', 1), ('both', 1), ('have', 1), ('their', 1), ('domestic', 1), ('audiences', 1), ('in', 1), ('mind', 1), ('will', 1), ('also', 1), ('want', 1), ('to', 1), ('apply', 1), ('public', 1), ('pressure', 1), ('on', 1), ('each', 1), ('other', 1)])

##### Task 3: build the vocabulary   **[20 marks]**
1. Process the documents one by one. That means, you call each file in the train-list (input to the function) after assigning the correct path.
2. Read the file and tokenize it using the function "*neg_modifier*".
3. Update the dictionary. The keys are tokens and the valuues are document counts.
4. The document count is the number of documents a token appeared in. If a token, say '*great*', appears three times in a document the document count increases by one.
5. The function does not return anything but fills up the dictionary *vocab_d* defined outside.
6. If your computer takes too much time for the following two functions take slices of train-list (and test-list). For example, if *tr3* is the full list of trainin file names, you may define $tr4 = tr3[:int(len(tr3)*.5)]$ (fifty percent of the files).
7. Do not modify the directories.
   

In [14]:
#function for creating the vocabulary dictionary
vocab_d = defaultdict(int) #dictionary for storing tokens and their document frequency
def build_vocab(tr):
    """Populate the global vocab_d dictionary using the training list file.

       Each key is a token and each value is the number of documents (DF) in
       which that token appears at least once.
    """
    global vocab_d

    for fname, label in tr:
        path = get_path(fname, label)
        try:
            with open(path, "r", encoding='utf-8', errors='replace') as fh:
              txt = fh.read()
        except OSError:
            continue # skip files that are unreadable

        tokens = neg_modifier(txt) # tokenise with negation handler

        # use set() to count token once per document (document frequency)
        for tok in set(tokens):
            vocab_d[tok] += 1
    return

In [15]:
#test your code
t0 = time.time()
build_vocab(tr3)
t1 = time.time()-t0

In [16]:
print(t1, len(vocab_d), len(tr3))

8.469696283340454 214076 34341


#### Task 4: complete the tf-idf matrix   **[25 marks]**
1. Process the documents one by one. That means, you call each file in the train-list after assigning the correct path.
2. Read the file and tokenize it using the function "*neg_modifier*".
3. Use the dictionary *vocab_d* for document frequency of tokens.
4. For term or token frequency use *add-1* smoothing. That means, the defalt count of a token in a document is 1.
5. The function *tf-idf_matrix*  takes three inputs and has two outputs.
6. The inputs are as follows:
   1. *train-data*: training/test file list
   2. *vocab_dict*: vocabulary dictionary
   3. *ts*: a keyword argument, if true, then the input is test file list
7. Note that a token amy appear in a test file that was not there in any file in train-list. It will be absent in the vocabulary dictionary.
8. The output of the function are, *X* and *y*. *X* represents the feature matrix and *y* the vector of labels.
   

#### Building the tf/idf matrix.

1. The following function is perhaps the most important for this assignment. Now that we have the vocabulary we create the tf-idf vector for each document.
2. This is the map taking a document to a vector.
3. The size of the vector will equal the length of the vocabulary.
4. Put the vector as a *row vector* in the input matrix.
5. Suppose the training set has $m$ documents and the vocabulary has $n$ tokens. The tf/idf matrix is of the order $m\times n$.
6. Remember each row represents a document.
7. You should simultaneously create the output vector $y$ representing the sentiment of the corresponding document. It is a column vector with dimension $m$. For example, a "train-set" with 4 files and vocabulary consisting of 5 tokens will have the form:
$$
X =
\begin{pmatrix}
x_{11} & x_{12} & x_{13} & x_{14} & x_{15} \\
x_{21} & x_{22} & x_{23} & x_{24} & x_{25} \\
x_{31} & x_{32} & x_{33} & x_{34} & x_{35} \\
x_{41} & x_{42} & x_{43} & x_{44} & x_{45}
\end{pmatrix}
\longrightarrow
y = \begin{pmatrix}
0\\
1\\
0\\
1
\end{pmatrix}
$$
8. Here the first row represents the tf-idf vector for document 1 which is negative (0), the second row for document 2 (positive) etc.    
9. Separate matrices must be created for training and test sets respectively.
10. **Do not use any ready-made library functions for tf-idf**.


In [17]:
# def tf_idf_matrix(in_data, vocab_dict, ts=0):
#     nrow = len(in_data)
#     ncol = len(vocab_dict)
#     X = np.zeros((nrow, ncol), dtype=np.float32)
#     y = np.zeros(nrow, dtype=np.int16)

#     # Build token -> column index mapping, dict lookup becomes O(1)
#     vocab_index = {tok: idx for idx, tok in enumerate(vocab_dict.keys())}

#     N = len(tr3) # number of training documents

#     for row_i, (fname, label) in enumerate(in_data):
#         path = get_path(fname, label)
#         try:
#             with open(path, 'r', encoding='utf-8', errors='replace') as fh:
#                 txt = fh.read()
#         except OSError:
#             y[row_i] = label
#             continue # If row unreadable leave it as zero

#         tokens = neg_modifier(txt)    # tokenise using negation handling
#         tf_counts = Counter(tokens)   # raw term frequencies in this doc

#         for tok, raw_tf in tf_counts.items():
#             if tok not in vocab_index:
#                 continue    # handles text encountered in test, previously not in train

#             col_j = vocab_index[tok]
#             df = vocab_dict[tok]    # document frequency from train dataset

#             # Add-1 smoothin on TF: count + 1
#             tf_smooth = raw_tf + 1

#             # IDF: log(N / df)
#             idf = np.log(N / df)

#             X[row_i, col_j] = tf_smooth * idf

#         y[row_i] = label    # Store label for document

#     return X, y

In [18]:

def tf_idf_matrix(in_data, vocab_dict, ts=0):
    """
       Build TF-IDF matrix X and label vector y from a file list.

       TF = add-1 smoothed term frequency, count(token in doc) + 1
       IDF = log( N_train / df(token)) N_train is len(vocab_dict)

       Returns a CSR sparse matrix instead of a dense ndarray.
       Memory: ~120 MB instead of ~12 GB for 20k docs × 150k features.
    """
    nrow  = len(in_data)
    N     = len(tr3)
    vocab_index = {tok: idx for idx, tok in enumerate(vocab_dict.keys())}

    # Accumulators for COO sparse format (row, col, value triples)
    rows, cols, vals = [], [], []
    y = np.zeros(nrow, dtype=np.int16)

    for row_i, (fname, label) in enumerate(in_data):
        path = get_path(fname, label)
        try:
            with open(path, 'r', encoding='utf-8', errors='replace') as fh:
                txt = fh.read()
        except OSError:
            y[row_i] = label
            continue

        tokens    = neg_modifier(txt)
        tf_counts = Counter(tokens)

        for tok, raw_tf in tf_counts.items():
            if tok not in vocab_index:
                continue
            col_j = vocab_index[tok]
            df    = vocab_dict[tok]
            tfidf = (raw_tf + 1) * np.log(N / df)

            # Only store non-zero entries — this is what makes it sparse
            rows.append(row_i)
            cols.append(col_j)
            vals.append(tfidf)

        y[row_i] = label

    # Build CSR matrix from COO triples
    X = sp.coo_matrix(
        (vals, (rows, cols)),
        shape=(nrow, len(vocab_dict)),
        dtype=np.float32,
    ).tocsr()   # CSR = fast row slicing, required by sklearn

    return X, y

In [19]:
#For testing
t0 = time.time()
X_tr, y_tr= tf_idf_matrix(tr3, vocab_d)
t1 = time.time()-t0

In [20]:
print(t1, X_tr.shape, y_tr.shape)

17.924118041992188 (34341, 214076) (34341,)


In [21]:
# print(f"Label distribution — neg: {(y_tr==0).sum()}, pos: {(y_tr==1).sum()}")

**Task 5**: Apply logistic regression from **sk-learn**  **[5 marks]**

1. There are 2 more functions for training and testing using **X** and **y** above.
2. You may use any intermediate "helper" functions if necessary.
3. The first is the library function below.


In [22]:
# for training data
param = None # Create a global variable to store trained model, to used by test_model.
def apply_logit(X, yt):
    """Apply scaling to features and train a logistic regression classification
       model.
    """
#Modify the following constructor, which builds a LogisticRegression object. You should change the default values.
#Please read the documentation in "sklearn".
    global param

    scaler = StandardScaler(with_mean=False) # with_mean=False preserves sparsity
    X_scaled = scaler.fit_transform(X)

    n_cores = max(1, mp.cpu_count() // 2)

    logit = LogisticRegression(
        solver='saga',
        C=1.0,
        max_iter=1000,
        random_state=42,
        n_jobs=n_cores,
        tol=1e-3,
    )
#Train the model and replace pass.
    logit.fit(X_scaled, yt)
#"clf" is the output of "logit" after training.

    param = (scaler, logit)
    clf = logit
    return clf

In [23]:
# Training the model
t0 = time.time()
clf = apply_logit(X_tr, y_tr)
time.time() - t0

33.877246141433716

In [24]:
clf.classes_, clf.n_iter_

(array([0, 1], dtype=int16), array([363], dtype=int32))

#### Task 6: test the model  [**10 marks**]

1. Test the model with the test data X, y.
2. You have to create the tf-idf features for the documents in the test files. Be careful while creating the feature vectors for test files. For testing you should change value of *ts*.
3. You should use the output of the previous function to predict the sentiment (= 0 or 1).
4. Hint: look at the "predict" function in LogisticRegression
5. Now compare the class prediction of the model *y_pred* with the actual value given by *y*.
6. Compute the performance metrics *accuracy*, *precision", *recall* and *F1-score* and report.
7. Do not use metrics/score from **sk-learn**


#### Metrics for binary classification     
For your convenience the definition of metrics given below.

#### Metrics
1. $\text{precision} = \frac{tp}{tp + fp} $
2. $\text{recall} = \frac{tp}{tp + fn} $
3. $\text{accuracy} = \frac{tp+tn}{tp +tn+ fp+fn} $
4. $$ F_\beta =\frac{(1 + \beta^2)\cdot tp}{(1 + \beta^2)\cdot tp + \beta^2\cdot fn + fp}$$
5. $$F_1 = \frac{tp}{tp + (fp+fn)/2}$$

1. $tp = \text{number of "true positive" } $
2. $tn = \text{number of "true negative" }$
3. $fp = \text{number of "false positive" }$
4. $fn = \text{number of "false negative" }$


In [26]:
# Build test matrix (ts=1 signals test mode — no refitting of IDF)
t0 = time.time()
X_ts, y_ts = tf_idf_matrix(ts3, vocab_d, ts=1)
t1 = time.time() - t0
t1, X_ts.shape, y_ts.shape

(4.564791440963745, (8586, 214076), (8586,))

In [28]:
def test_model(X, y):
    """
    Evaluate the trained model on a held-out feature matrix X and labels y.

    Use the global 'param' tuple (scaler, clf) produced by apply_logit.'

    Metrics are computed manually from tp/tn/fp/fn counts
    """
    #use the object "param" above, this is the output after training
    global param
    scaler, clf = param

    acc = 0
    prec = 0
    recall = 0
    f1 = 0

    # Apply the same scaling fitted on training data (no re-fitting)
    X_scaled = scaler.transform(X)

    y_pred = clf.predict(X_scaled)   # binary predictions: 0 or 1

    #  Confusion matrix counts (positive class = 1)
    tp = int(np.sum((y_pred == 1) & (y == 1)))   # predicted pos, actual pos
    tn = int(np.sum((y_pred == 0) & (y == 0)))   # predicted neg, actual neg
    fp = int(np.sum((y_pred == 1) & (y == 0)))   # predicted pos, actual neg
    fn = int(np.sum((y_pred == 0) & (y == 1)))   # predicted neg, actual pos

    # Compute performance metrics
    acc    = (tp + tn) / (tp + tn + fp + fn)
    prec   = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1     = tp / (tp + (fp + fn) / 2) if (tp + (fp + fn) / 2) > 0 else 0.0

    return acc, prec, recall, f1


acc, prec, recall, f1 = test_model(X_ts, y_ts)

print("Test Set Performance")
print(f"Accuracy  : {acc:.4f}")
print(f"Precision : {prec:.4f}")
print(f"Recall    : {recall:.4f}")
print(f"F1 Score  : {f1:.4f}")

Test Set Performance
Accuracy  : 0.8962
Precision : 0.9042
Recall    : 0.8877
F1 Score  : 0.8959


#### Task 7: report and bonus [**8+7 marks**]
1. This assignment requires handling of a relatively large set of text files.
2. Some functions have to access the disk many times to load and read the files.
3. Moreover, the matrices/tensors are large.
4. The questions you should think about and discuss briefly:
    1. memory requirements
    2. parallelization
    3. Python modules for parallel processing
5. The 7 bonus marks are for attempts at modifying the functions in **task 3** and/or **task 4** using parallel processing Python libraries.

In [ ]:
def _vocab_worker(file_chunk):
    """
      Worker: build a partial vocabulary dict from a chunk of (fname, label) pairs.
      Returns a plain dict (must be picklable to cross process boundary).
    """
    partial_vocab = defaultdict(int)
    for fname, label in file_chunk:
        path = get_path(fname, label)
        try:
            with open(path, 'r', encoding='utf-8', errors='replace') as fh:
                txt = fh.read()
        except OSError:
            continue
        for tok in set(neg_modifier(txt)):
            partial_vocab[tok] += 1
    return dict(partial_vocab)


def build_vocab_parallel(tr, n_workers=None):
    """
        Parallel version of build_vocab using multiprocessing.Pool.

        Splits the file list into n_workers chunks, processes each chunk in a
        separate process, then merges the partial vocabulary dicts.

        Args:
          tr        : list of (filename, label) tuples
          n_workers : int — number of parallel processes (default: cpu_count)
    """
    global vocab_d
    if n_workers is None:
        n_workers = mp.cpu_count()

    # Split file list into almost equal chunks for each worker
    chunk_size = max(1, len(tr) // n_workers)
    chunks = [tr[i:i + chunk_size] for i in range(0, len(tr), chunk_size)]

    vocab_d = defaultdict(int)   # reset global

    with mp.Pool(processes=n_workers) as pool:
        partial_results = pool.map(_vocab_worker, chunks)

    # Merge partial dicts: sum document frequencies across all workers
    for partial in partial_results:
        for tok, df in partial.items():
            vocab_d[tok] += df

    return


# Benchmark parallel vs sequential
n_cpu = mp.cpu_count()
# print(f"CPU cores available: {n_cpu}")

vocab_d = defaultdict(int)   # reset
t0 = time.time()
build_vocab_parallel(tr3, n_workers=n_cpu)
time.time() - t0
# print(f"Parallel vocab build: {t_parallel:.2f}s | vocab size: {len(vocab_d):,}")

In [ ]:
def _tfidf_worker(args):
    """
      Worker: compute TF-IDF rows for a chunk of documents.

      Returns a list of (row_index, sparse_dict, label) tuples where
      sparse_dict maps column index → tfidf value (only non-zero entries).
      Returning sparse dicts avoids sending large zero-padded arrays over IPC.
    """
    chunk, vocab_dict, N, start_row = args
    vocab_index = {tok: idx for idx, tok in enumerate(vocab_dict.keys())}

    rows = []
    for local_i, (fname, label) in enumerate(chunk):
        path = get_path(fname, label)
        try:
            with open(path, 'r', encoding='utf-8', errors='replace') as fh:
                txt = fh.read()
        except OSError:
            rows.append((start_row + local_i, {}, label))
            continue

        tokens = neg_modifier(txt)
        tf_counts = Counter(tokens)
        sparse_row = {}

        for tok, raw_tf in tf_counts.items():
            if tok not in vocab_index:
                continue
            col_j  = vocab_index[tok]
            df     = vocab_dict[tok]
            tf_s   = raw_tf + 1          # add-1 smoothing
            idf    = np.log(N / df)
            sparse_row[col_j] = tf_s * idf

        rows.append((start_row + local_i, sparse_row, label))
    return rows


def tf_idf_matrix_parallel(in_data, vocab_dict, ts=0, n_workers=None):
    """
    Parallel version of tf_idf_matrix.

    Each worker handles a chunk of documents and returns sparse row dicts.
    The main process assembles the final dense matrix.
    """
    if n_workers is None:
        n_workers = mp.cpu_count()

    nrow = len(in_data)
    ncol = len(vocab_dict)
    N    = len(tr3)

    chunk_size = max(1, nrow // n_workers)
    chunks = [
        (in_data[i:i+chunk_size], vocab_dict, N, i)
        for i in range(0, nrow, chunk_size)
    ]

    X = np.zeros((nrow, ncol), dtype=np.float32)
    y = np.zeros(nrow, dtype=np.int16)

    with mp.Pool(processes=n_workers) as pool:
        all_row_results = pool.map(_tfidf_worker, chunks)

    # Assemble sparse results into dense matrix
    for chunk_result in all_row_results:
        for global_row, sparse_dict, label in chunk_result:
            for col_j, val in sparse_dict.items():
                X[global_row, col_j] = val
            y[global_row] = label

    return X, y


print("Building TF-IDF matrix in parallel...")
t0 = time.time()
X_tr_p, y_tr_p = tf_idf_matrix_parallel(tr3, vocab_d, n_workers=n_cpu)
print(f"Parallel matrix build: {time.time()-t0:.2f}s | shape: {X_tr_p.shape}")

# Verify parallel and sequential produce same results (within float32 tolerance)
print(f"Max difference from sequential: {np.max(np.abs(X_tr - X_tr_p)):.6f}")

### Model Summary

This sentiment analysis framework leverages a robust **bag-of-words and TF-IDF pipeline paired with Logistic Regression** to classify binary sentiment in IMDB movie reviews. Designed for immense scalability, the system processes documents sequentially, ensuring that memory consumption is restricted strictly to the feature matrix and vocabulary rather than the entire raw text corpus.

A standout feature of this architecture is its **sophisticated negation handling**. By attaching a "NOT_" prefix to tokens that follow a negation trigger up until the next punctuation mark, the model effectively distinguishes between contradictory sentiments, such as "like" and "NOT_like". This targeted approach significantly boosts the model's recall capabilities when dealing with inverted sentiments. Furthermore, the TF-IDF mechanism promotes highly discriminative tokens while applying Add-1 smoothing to regularize representations and prevent zero weights.

To manage the immense sparsity of the resulting feature matrix—which contains up to 200,000 unique tokens and consists of over 99% zeros—the system employs **`scipy.sparse` matrices**. Storing only non-zero entries mitigates what would otherwise be a crippling 12 GB RAM requirement for a 20,000-document dataset, reducing memory usage by approximately 100 times. Because review lengths vary drastically, a `StandardScaler` is also applied to normalize term frequency magnitudes before passing them to the solver.

Finally, the framework embraces **parallelization** to handle resource-heavy tokenization and disk I/O operations. By utilizing Python's `multiprocessing.Pool.map()` to distribute tasks across multiple workers and minimize inter-process communication overhead, the pipeline achieves near-linear speedups. With increased data Apache Spark becomes a good option for better multiprocessing worloads.
